# Tree-automatic structures and uniformly tree-automatic classes

An *automatic* presentation encodes elements as **strings** and recognizes the
relations with word automata — that is Büchi's theorem, made executable. A
**tree-automatic** presentation encodes elements as finite **trees** and uses
bottom-up tree automata; that is Rabin's.

Trees buy expressive power. This notebook shows three things you cannot do (or
cannot do naturally) with words:

1. **Skolem arithmetic** $(\mathbb{N}_{>0}, \cdot)$ — a number is the tree of its prime exponents.
2. **Bounded tree-width graphs** — the advice is a tree decomposition.
3. **Bounded clique-width graphs** — the advice is a $k$-expression.

For 2 and 3, a monadic second-order query is compiled **once for the whole class**
and then decides the property on any member in time linear in its advice tree.
That is Courcelle's theorem, with the automaton actually built.

## 1. Skolem arithmetic: multiplication over the positive integers

$(\mathbb{N}_{>0}, \cdot)$ is *not* automatic over strings, but it is
tree-automatic: write $n = \prod_i p_i^{e_i}$ as a spine of primes with each
exponent hanging off as a binary chain, and multiplication becomes componentwise
addition along the branches.

In [ ]:
from autstr.buildin.tree_presentations import skolem_arithmetic

skolem = skolem_arithmetic()
n = 360                                     # 2^3 * 3^2 * 5
tree = skolem.encode(n)
print("360 decodes back to:", skolem.decode(tree))

In [ ]:
# M(x, y, z) is x * y = z -- one automaton, an infinite domain
M = skolem.evaluate('M(x,y,z)')
enc = skolem.encode
for x, y, z in [(6, 10, 60), (7, 13, 91), (6, 11, 65)]:
    print(f"{x} * {y} = {z}?  {M.accepts(enc(x), enc(y), enc(z))}")

In [ ]:
# Divisibility is first-order over multiplication alone:
#   x | y   iff   exists z. x * z = y
divides = skolem.evaluate('exists z.(M(x,z,y))')
for x, y in [(3, 12), (5, 12), (7, 91)]:
    print(f"{x} divides {y}: {divides.accepts(enc(x), enc(y))}")

## 2. Bounded tree-width: connectedness, compiled once

`TreeWidthClass(w)` presents *every* graph of tree-width $\le w$ at once. Elements
are **sets of vertices**, so first-order logic over the presentation is monadic
second-order logic over the graph.

In [ ]:
from autstr.tree_graphs import TreeWidthClass, TreeWidthGraph
import networkx as nx

CONNECTED = ('all c.(((exists x.(Sing(x) and Subset(x,c))) and '
             '(all x.(all y.((Subset(x,c) and E(x,y)) -> Subset(y,c))))) '
             '-> (all x.(Sing(x) -> Subset(x,c))))')

tw = TreeWidthClass(1)                       # forests
automaton, tapes = tw.evaluate(CONNECTED)    # ONE compile for the whole class
print(f"{automaton.num_states}-state tree automaton, tapes = {tapes}")

In [ ]:
# ... and now it decides connectedness on any forest, in linear time.
path   = TreeWidthGraph.from_networkx(nx.path_graph(6))
forest = TreeWidthGraph.from_networkx(nx.disjoint_union(nx.path_graph(3),
                                                        nx.path_graph(3)))
for name, g in [("path P6", path), ("two disjoint P3", forest)]:
    print(f"{name:18s} connected = {tw.check(CONNECTED, g)}")

## 3. Bounded clique-width: a k-expression as advice

A $k$-expression builds a graph from labelled vertices with three operations:
disjoint union `u`, relabel `r{i}{j}`, and join `e{i}{j}` (every label-$i$ vertex
to every label-$j$ vertex). Adjacency is cheap to recognize — two vertices are
joined exactly when some `e{i}{j}` node sees them holding its two labels — so MSO
queries compile far faster here than for tree-width.

In [ ]:
from autstr.tree_graphs import CliqueWidthClass, CliqueWidthGraph

K4 = CliqueWidthGraph.clique(4)
print("K4 edges:", sorted(tuple(sorted(e)) for e in K4.edges))
print("expression leaves are the vertices:", K4.vertices)

In [ ]:
TWO_COL = ('exists c.(all x.(all y.(E(x,y) -> '
           '(not ((Subset(x,c) and Subset(y,c)) or '
           '((not Subset(x,c)) and (not Subset(y,c))))))))')

import time
cw = CliqueWidthClass(2)
t = time.time()
two_col, _ = cw.evaluate(TWO_COL)
print(f"2-colourability: {two_col.num_states} states, "
      f"compiled in {time.time()-t:.1f}s")

In [ ]:
# One automaton, checked against networkx on every member we can name.
cases = [("K2", CliqueWidthGraph.clique(2)),
         ("K3", CliqueWidthGraph.clique(3)),
         ("K4", CliqueWidthGraph.clique(4)),
         ("K2,3", CliqueWidthGraph.complete_bipartite(2, 3)),
         ("K3,3", CliqueWidthGraph.complete_bipartite(3, 3))]
print(f"{'graph':6s} {'autstr':>8s} {'networkx':>10s}")
for name, g in cases:
    ours = cw.check(TWO_COL, g)
    theirs = nx.is_bipartite(g.to_networkx())
    assert ours == theirs
    print(f"{name:6s} {ours!s:>8s} {theirs!s:>10s}")

## Where the cost lives

Deciding is always linear in the advice tree — about 150–190 ns per node for both
classes. What differs is the **compile**: clique-width's small alphabet and simple
edge automaton make MSO queries cheap, while tree-width's edge automaton carries
pending registers over a wider alphabet.

`benchmarks/tree_class_benchmark.py` measures both, with a light and a heavy
profile.